# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`This notebook guides you through loading, exploring, and processing a Croissant dataset using the `mlcroissant` library. You will use the FAIR² Mental Health Survey dataset from Kilifi County, Kenya, which contains survey-based mental health indicators, demographic details, and standardized psychological scores.### Dataset SourceThe dataset is described by the Croissant schema at:`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pd# Define the dataset URLurl = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'# Load the dataset metadatadataset = mlc.Dataset(url)print(f"{dataset.metadata.name}: {dataset.metadata.description}")print(f"Version: {dataset.metadata.version}")print(f"Date Published: {dataset.metadata.datePublished}")print(f"License URL: {dataset.metadata.license}")

## 2. Data OverviewReview available record sets, fields, columns, and their unique `@id`s.The Croissant schema provides structured record sets. Here we enumerate the available record sets, their fields, and columns, referencing all by their `@id` for consistency.

In [ ]:
# List all available record sets with their @id and fieldsrecord_sets = dataset.metadata.recordSetif not record_sets:    print("No record sets found in metadata.")else:    for rs in record_sets:        print(f"RecordSet Name: {getattr(rs, 'name', 'N/A')} | RecordSet @id: {rs['@id']}")        if hasattr(rs, 'field'):            print("  Fields:")            fields = rs.field            for f in fields:                print(f"    {getattr(f, 'name', 'N/A')} (@id: {f['@id']}) | Type: {getattr(f, 'dataType', 'N/A')}")        if hasattr(rs, 'column'):            print("  Columns:")            columns = rs.column            for c in columns:                print(f"    {getattr(c, 'name', 'N/A')} (@id: {c['@id']})")

## 3. Data ExtractionLoad data from a specific record set into a DataFrame for analysis.Below you'll extract and examine all records from each record set, referencing each set by its `@id`.

In [ ]:
# Extract data from each record set referenced by @id# Gather record set @id's dynamically from metadatarecordset_ids = []if record_sets:    for rs in record_sets:        recordset_ids.append(rs['@id'])else:    # If no record sets found, you can manually specify the @id(s) from documentation.    recordset_ids = []dataframes = {}for rs_id in recordset_ids:    print(f"Loading records for RecordSet @id: {rs_id}")    records = list(dataset.records(record_set=rs_id))    if records:        df = pd.DataFrame(records)        dataframes[rs_id] = df        print(f"Fields (@id) in this RecordSet: {df.columns.tolist()}")        print(df.head())    else:        print(f"No records found for RecordSet @id: {rs_id}")

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All entities are referenced by `@id`.Here, select a numeric or key survey field for filtering and normalization, referencing its `@id`. Then explore grouping by a demographic attribute.

In [ ]:
# Example: Filter, normalize, and group using column @id# First, select a record set with substantial numeric survey fields.# For demonstration, assuming the main survey responses record set has @id:# 'https://api.app.sen.science/frontiers/7160411/mental-health-survey-recordset'# Replace with the actual @id(s) listed in Section 2 above.if dataframes:    # Pick the first available record set    main_rs_id = list(dataframes.keys())[0]    df = dataframes[main_rs_id]    # Identify a numeric field, e.g., GAD-7 score, by its column @id    # For example: 'gad7_score', or its Croissant @id    # Attempt to infer numeric fields from schema or printed columns above    potential_numeric_fields = [col for col in df.columns if 'gad' in col or 'phq' in col or 'psq' in col or 'score' in col or 'Age' in col]    numeric_field_id = potential_numeric_fields[0] if potential_numeric_fields else df.columns[0]  # default fallback    print(f"Using numeric field: {numeric_field_id}")    # Filter records with the numeric field above a threshold    threshold = 10    filtered_df = df[df[numeric_field_id] > threshold]    print(f"Filtered records with {numeric_field_id} > {threshold}:")    print(filtered_df.head())    # Normalize the numeric field    norm_col = f"{numeric_field_id}_normalized"    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()    print(f"Normalized {numeric_field_id} for filtered records:")    print(filtered_df[[numeric_field_id, norm_col]].head())    # Group by a demographic field, e.g., 'gender' or its @id    potential_group_fields = [col for col in df.columns if 'gender' in col.lower() or 'education' in col.lower() or 'village' in col.lower()]    group_field = potential_group_fields[0] if potential_group_fields else df.columns[0]    if group_field in filtered_df.columns:        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)        print(f"Grouped data by {group_field}:")        print(grouped_df.head())else:    print("No extracted dataframes available for EDA.")

## 5. VisualizationVisualize data distributions or relationships between fields in the dataset.Let's plot the distribution of a numeric field and compare averages across demographic groups.

In [ ]:
import matplotlib.pyplot as pltif dataframes:    df = dataframes[main_rs_id]    # Histogram of the numeric field    plt.figure(figsize=(8,4))    df[numeric_field_id].hist(bins=20)    plt.title(f"Distribution of {numeric_field_id}")    plt.xlabel(numeric_field_id)    plt.ylabel("Count")    plt.show()    # Bar plot of normalized scores by group_field    if group_field in df.columns and norm_col in filtered_df.columns:        plt.figure(figsize=(8,5))        means = filtered_df.groupby(group_field)[norm_col].mean()        means.plot(kind='bar')        plt.ylabel(f"Mean of {norm_col}")        plt.xlabel(group_field)        plt.title(f"Average Normalized {numeric_field_id} by {group_field}")        plt.show()else:    print("Visualization skipped: no dataframes loaded.")

## 6. ConclusionThis notebook demonstrated how to load, explore, and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya as defined by a Croissant schema. Using the `mlcroissant` library, you can flexibly extract records by their `@id`, analyze demographic and mental health fields, and visualize data distributions. The dataset supports study of mental health trends, demographic associations, and enables AI-ready standards for open health research in Africa.For deeper analysis, reference specific field and column `@id`s from the metadata, and extend EDA as needed.